In [3]:
import tensorflow as tf
import numpy as np

# Load the saved Keras model
model = tf.keras.models.load_model('0210.keras')

# Define a representative dataset generator for quantization calibration
# Option 1: Use random data (if X is not available)
# def representative_dataset():
#     for _ in range(100):  # Use 100-500 samples for calibration
#         data = np.random.rand(1, 10, 3).astype(np.float32)
#         yield [data]

# Option 2: Use a subset of your actual training data (X) for better accuracy (uncomment if available)
# def representative_dataset():
#     for data in X[:100]:  # Adjust slice as needed
#         yield [data.reshape(1, 10, 3).astype(np.float32)]

# Create the TFLite converter
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Enable quantization
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS  # 如果需要 TensorFlow 算子，取消注释
]

# # Ensure full integer quantization (int8 for inputs/outputs)
# converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
# converter.inference_input_type = tf.int8
# converter.inference_output_type = tf.int8

# Convert the model
tflite_model = converter.convert()

# Save the quantized TFLite model
with open('0210_quantized.tflite', 'wb') as f:
    f.write(tflite_model)

print("Quantized TFLite model saved as '0210_quantized.tflite'")

INFO:tensorflow:Assets written to: C:\Users\y2451\AppData\Local\Temp\tmp58n623af\assets


INFO:tensorflow:Assets written to: C:\Users\y2451\AppData\Local\Temp\tmp58n623af\assets


Saved artifact at 'C:\Users\y2451\AppData\Local\Temp\tmp58n623af'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 10, 3), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  1622804363984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1622804363408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1622804364752: TensorSpec(shape=(), dtype=tf.resource, name=None)
Quantized TFLite model saved as '0210_quantized.tflite'


In [4]:
for layer in model.layers:
    print(f"\n层: {layer.name}")
    for weight in layer.weights:
        print(f"  {weight.name}: {weight.shape}")


层: lstm
  kernel: (3, 64)
  recurrent_kernel: (16, 64)

层: dense
  kernel: (16, 2)


In [ ]:
import os

def convert_to_c_array(tflite_path, header_path):
    with open(tflite_path, 'rb') as f:
        model_data = f.read()
    
    c_array = ', '.join(f'0x{b:02x}' for b in model_data)
    
    header_content = f"""
#ifndef MODEL_DATA_H
#define MODEL_DATA_H

const unsigned char g_model[] = {{
    {c_array}
}};

const int g_model_len = {len(model_data)};

#endif // MODEL_DATA_H
"""
    
    with open(header_path, 'w') as f:
        f.write(header_content)
    
    print(f"Model converted: {len(model_data)} bytes")

# 使用
convert_to_c_array('0210_quantized.tflite', 'model_data.h')